In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2009_2010"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "OSQ_F.xpt"
csv_file = DATA_DIR / "OSQ_F_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded XPT values.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2009-2010 OSQ_F XPT -> CSV")
print("=" * 75)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "OSQ010A",
    "OSQ010B",
    "OSQ010C",
    "OSQ020A",
    "OSQ020B",
    "OSQ020C",
    "OSD030AA",
    "OSQ040AA",
    "OSD050AA",
    "OSD030AB",
    "OSD050AB",
    "OSD030BA",
    "OSQ040BA",
    "OSD050BA",
    "OSD030BB",
    "OSQ040BB",
    "OSD050BB",
    "OSD030BC",
    "OSQ040BC",
    "OSD050BC",
    "OSD030BD",
    "OSQ040BD",
    "OSD050BD",
    "OSD030BE",
    "OSQ040BE",
    "OSD030BF",
    "OSQ040BF",
    "OSD030BG",
    "OSQ040BG",
    "OSD030BH",
    "OSQ040BH",
    "OSD030BI",
    "OSQ040BI",
    "OSD030BJ",
    "OSQ040BJ",
    "OSD030CA",
    "OSQ040CA",
    "OSD050CA",
    "OSD030CB",
    "OSQ040CB",
    "OSD050CB",
    "OSD030CC",
    "OSQ040CC",
    "OSD050CC",
    "OSQ080",
    "OSQ090A",
    "OSQ100A",
    "OSD110A",
    "OSQ120A",
    "OSQ090B",
    "OSQ100B",
    "OSD110B",
    "OSQ120B",
    "OSQ090C",
    "OSQ100C",
    "OSD110C",
    "OSQ120C",
    "OSQ090D",
    "OSQ100D",
    "OSD110D",
    "OSQ120D",
    "OSQ090E",
    "OSQ100E",
    "OSD110E",
    "OSQ120E",
    "OSQ090F",
    "OSQ100F",
    "OSD110F",
    "OSQ120F",
    "OSQ090G",
    "OSQ120G",
    "OSQ090H",
    "OSQ120H",
    "OSQ090I",
    "OSQ120I",
    "OSQ090J",
    "OSQ120J",
    "OSQ060",
    "OSQ070",
    "OSQ130",
    "OSQ140Q",
    "OSQ140U",
    "OSQ150",
    "OSQ160A",
    "OSQ160B",
    "OSQ170",
    "OSQ180",
    "OSQ190",
    "OSQ200",
    "OSQ210",
    "OSQ220"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "OSQ_F column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 6218:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) != 92:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,218 rows x 92 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# IMPORTANT:
#
# Never globally convert this value to zero.
#
# For the attached OSQ_F:
#
#     tiny artifacts = 0
#
# Therefore NO zero restoration is required.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_total = int(
    df[numeric_cols]
    .eq(0)
    .sum()
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "True numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "were found in OSQ_F. "
        "Do not automatically replace them."
    )


print(
    "PASS: No tiny XPORT zero-decoding artifacts."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# CRITICAL FIDELITY RULE:
#
# Never round genuine decimal values.
#
# Always perform this check BEFORE converting
# anything to Int64.
#
# The attached OSQ_F contains:
#
#     genuine decimal variables = NONE
#
# If a different copy unexpectedly contains fractions,
# the program will STOP instead of rounding them.
#
# This same safeguard protects:
#
#     Dietary decimals
#     Laboratory measurements
#     Examination measurements
#     Sample weights
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "genuine fractional values"
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal variables detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values "
    "exist in OSQ_F."
)


# ============================================================
# 6. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


if (
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
    or
    df["SEQN"].min()
    != 51624
    or
    df["SEQN"].max()
    != 62160
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 7. FORMAT ALL PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# OSQ_F has no genuine fractional variables.
#
# Therefore all 92 released fields can safely
# be converted to nullable Int64.
#
# IMPORTANT:
#
# NO .round() is used.
#

for col in df.columns:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_number_check = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not np.all(
            whole_number_check
        ):

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 92 OSQ_F variables formatted "
    "without unnecessary .0."
)


# ============================================================
# 8. CDC CHECK - OSQ010A/B/C
# ============================================================

#
# Hip fracture:
#
# 1 Yes         = 90
# 2 No          = 6123
# 9 Don't know  = 5
#
#
# Wrist fracture:
#
# 1 Yes         = 507
# 2 No          = 5707
# 9 Don't know  = 4
#
#
# Spine fracture:
#
# 1 Yes         = 111
# 2 No          = 6103
# 9 Don't know  = 4
#

CDC_FRACTURE_HISTORY = {

    "OSQ010A": {
        "values": {
            1: 90,
            2: 6123,
            9: 5
        },
        "missing": 0
    },

    "OSQ010B": {
        "values": {
            1: 507,
            2: 5707,
            9: 4
        },
        "missing": 0
    },

    "OSQ010C": {
        "values": {
            1: 111,
            2: 6103,
            9: 4
        },
        "missing": 0
    }
}


print("\n--- CDC FRACTURE HISTORY CHECKS ---")


for col, expected in CDC_FRACTURE_HISTORY.items():

    actual_values = (
        df[col]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if actual_values != expected["values"]:

        raise ValueError(
            f"{col} frequencies differ from CDC.\n"
            f"Expected: {expected['values']}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected["missing"]:

        raise ValueError(
            f"{col} missing count differs from CDC."
        )


    print(
        f"PASS: {col} matches CDC."
    )


# ============================================================
# 9. CDC CHECK - OSQ020A/B/C
# ============================================================

#
# Number of fractures.
#

print("\n--- CDC FRACTURE COUNT CHECKS ---")


#
# Hip:
# 1-2 = 90
# Missing = 6128
#

s = df["OSQ020A"]


if not (
    int(
        s.between(
            1,
            2
        ).sum()
    )
    == 90
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 0
    and
    int(
        s.isna()
        .sum()
    )
    == 6128
):

    raise ValueError(
        "OSQ020A differs from CDC."
    )


print(
    "PASS: OSQ020A matches CDC."
)


#
# Wrist:
#
# 1-14 = 506
# 9999 = 1
# Missing = 5711
#

s = df["OSQ020B"]


if not (
    int(
        s.between(
            1,
            14
        ).sum()
    )
    == 506
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 1
    and
    int(
        s.isna()
        .sum()
    )
    == 5711
):

    raise ValueError(
        "OSQ020B differs from CDC."
    )


print(
    "PASS: OSQ020B matches CDC."
)


#
# Spine:
#
# 1-3 = 111
# Missing = 6107
#

s = df["OSQ020C"]


if not (
    int(
        s.between(
            1,
            3
        ).sum()
    )
    == 111
    and
    int(
        s.eq(7777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(9999)
        .sum()
    )
    == 0
    and
    int(
        s.isna()
        .sum()
    )
    == 6107
):

    raise ValueError(
        "OSQ020C differs from CDC."
    )


print(
    "PASS: OSQ020C matches CDC."
)


# ============================================================
# 10. CDC CHECK - FIRST HIP FRACTURE
# ============================================================

#
# OSD030AA:
#
# Age at first hip fracture
#
# 1-79 = 78
# 80+  = 12
# Missing = 6128
#

s = df["OSD030AA"]


print("\n--- OSD030AA ---")


if not (
    int(
        s.between(
            1,
            79
        ).sum()
    )
    == 78
    and
    int(
        s.eq(80)
        .sum()
    )
    == 12
    and
    int(
        s.eq(77777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(99999)
        .sum()
    )
    == 0
    and
    int(
        s.isna()
        .sum()
    )
    == 6128
):

    raise ValueError(
        "OSD030AA differs from CDC."
    )


print(
    "PASS: OSD030AA matches CDC."
)


# ============================================================
# 11. CDC CHECK - OSQ040AA
# ============================================================

#
# Under/over age 50 when first hip fracture:
#
# 1 = Under 50 = 48
# 2 = 50+      = 42
# Missing = 6128
#

EXPECTED_OSQ040AA = {
    1: 48,
    2: 42
}


actual = (
    df["OSQ040AA"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_OSQ040AA
    or
    int(
        df["OSQ040AA"]
        .isna()
        .sum()
    )
    != 6128
):

    raise ValueError(
        "OSQ040AA differs from CDC."
    )


print(
    "PASS: OSQ040AA matches CDC."
)


# ============================================================
# 12. CDC CHECK - OSD050AA
# ============================================================

#
# Reason first hip fracture:
#
# 1 = Fall from standing height = 27
# 2 = Hard fall                 = 13
# 3 = Severe trauma             = 2
# Missing = 6176
#

EXPECTED_OSD050AA = {
    1: 27,
    2: 13,
    3: 2
}


actual = (
    df["OSD050AA"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_OSD050AA
    or
    int(
        df["OSD050AA"]
        .isna()
        .sum()
    )
    != 6176
):

    raise ValueError(
        "OSD050AA differs from CDC."
    )


print(
    "PASS: OSD050AA matches CDC."
)


# ============================================================
# 13. CDC CHECK - OSQ080
# ============================================================

#
# Doctor ever said participant had any other fracture:
#
# 1 = Yes         1307
# 2 = No          4907
# 9 = Don't know     4
#

EXPECTED_OSQ080 = {
    1: 1307,
    2: 4907,
    9: 4
}


actual = (
    df["OSQ080"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_OSQ080
    or
    int(
        df["OSQ080"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "OSQ080 differs from CDC."
    )


print(
    "PASS: OSQ080 matches CDC."
)


# ============================================================
# 14. CDC CHECK - OSQ060
# ============================================================

#
# Ever told had osteoporosis / brittle bones:
#
# 1 = Yes          348
# 2 = No          5853
# 9 = Don't know    17
#

EXPECTED_OSQ060 = {
    1: 348,
    2: 5853,
    9: 17
}


actual = (
    df["OSQ060"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_OSQ060
    or
    int(
        df["OSQ060"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "OSQ060 differs from CDC."
    )


print(
    "PASS: OSQ060 matches CDC."
)


# ============================================================
# 15. CDC CHECK - OSQ070
# ============================================================

#
# Ever treated for osteoporosis:
#
# 1 = Yes = 228
# 2 = No  = 120
# Missing = 5870
#

EXPECTED_OSQ070 = {
    1: 228,
    2: 120
}


actual = (
    df["OSQ070"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_OSQ070
    or
    int(
        df["OSQ070"]
        .isna()
        .sum()
    )
    != 5870
):

    raise ValueError(
        "OSQ070 differs from CDC."
    )


print(
    "PASS: OSQ070 matches CDC."
)


# ============================================================
# 16. CDC CHECK - OSQ130
# ============================================================

#
# Ever taken prednisone / cortisone nearly daily:
#
# 1 = Yes          279
# 2 = No          5896
# 7 = Refused        1
# 9 = Don't know    42
#

EXPECTED_OSQ130 = {
    1: 279,
    2: 5896,
    7: 1,
    9: 42
}


actual = (
    df["OSQ130"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_OSQ130
    or
    int(
        df["OSQ130"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "OSQ130 differs from CDC."
    )


print(
    "PASS: OSQ130 matches CDC."
)


# ============================================================
# 17. CDC CHECK - OSQ140Q
# ============================================================

#
# Duration using prednisone / cortisone:
#
# 1-66 = 272
# 999  = 7
# Missing = 5939
#

s = df["OSQ140Q"]


if not (
    int(
        s.between(
            1,
            66
        ).sum()
    )
    == 272
    and
    int(
        s.eq(777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(999)
        .sum()
    )
    == 7
    and
    int(
        s.isna()
        .sum()
    )
    == 5939
):

    raise ValueError(
        "OSQ140Q differs from CDC."
    )


print(
    "PASS: OSQ140Q matches CDC."
)


# ============================================================
# 18. CDC CHECK - OSQ140U
# ============================================================

#
# Unit:
#
# 1 = Months = 183
# 2 = Years  = 89
# Missing = 5946
#

EXPECTED_OSQ140U = {
    1: 183,
    2: 89
}


actual = (
    df["OSQ140U"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_OSQ140U
    or
    int(
        df["OSQ140U"]
        .isna()
        .sum()
    )
    != 5946
):

    raise ValueError(
        "OSQ140U differs from CDC."
    )


print(
    "PASS: OSQ140U matches CDC."
)


# ============================================================
# 19. CDC CHECK - OSQ150
# ============================================================

#
# Either biological parent told had osteoporosis:
#
# 1 = Yes          525
# 2 = No          5408
# 7 = Refused        2
# 9 = Don't know   283
#

EXPECTED_OSQ150 = {
    1: 525,
    2: 5408,
    7: 2,
    9: 283
}


actual = (
    df["OSQ150"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_OSQ150
    or
    int(
        df["OSQ150"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "OSQ150 differs from CDC."
    )


print(
    "PASS: OSQ150 matches CDC."
)


# ============================================================
# 20. CDC CHECK - OSQ160A / OSQ160B
# ============================================================

#
# CODE ALL THAT APPLY:
#
# Mother = 494
# Father = 48
#

if not (
    df["OSQ160A"]
    .value_counts()
    .to_dict()
    == {
        1: 494
    }
    and
    int(
        df["OSQ160A"]
        .isna()
        .sum()
    )
    == 5724
):

    raise ValueError(
        "OSQ160A differs from CDC."
    )


print(
    "PASS: OSQ160A matches CDC."
)


if not (
    df["OSQ160B"]
    .value_counts()
    .to_dict()
    == {
        2: 48
    }
    and
    int(
        df["OSQ160B"]
        .isna()
        .sum()
    )
    == 6170
):

    raise ValueError(
        "OSQ160B differs from CDC."
    )


print(
    "PASS: OSQ160B matches CDC."
)


# ============================================================
# 21. CDC CHECK - MATERNAL HIP FRACTURE
# ============================================================

EXPECTED_OSQ170 = {
    1: 295,
    2: 5724,
    7: 1,
    9: 198
}


actual = (
    df["OSQ170"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_OSQ170
    or
    int(
        df["OSQ170"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "OSQ170 differs from CDC."
    )


print(
    "PASS: OSQ170 matches CDC."
)


#
# Mother's age when hip fractured:
#
# 10-100 = 289
# 999    = 6
# Missing = 5923
#

s = df["OSQ180"]


if not (
    int(
        s.between(
            10,
            100
        ).sum()
    )
    == 289
    and
    int(
        s.eq(777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(999)
        .sum()
    )
    == 6
    and
    int(
        s.isna()
        .sum()
    )
    == 5923
):

    raise ValueError(
        "OSQ180 differs from CDC."
    )


print(
    "PASS: OSQ180 matches CDC."
)


# ============================================================
# 22. CDC CHECK - PATERNAL HIP FRACTURE
# ============================================================

EXPECTED_OSQ200 = {
    1: 109,
    2: 5766,
    7: 4,
    9: 339
}


actual = (
    df["OSQ200"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if (
    actual != EXPECTED_OSQ200
    or
    int(
        df["OSQ200"]
        .isna()
        .sum()
    )
    != 0
):

    raise ValueError(
        "OSQ200 differs from CDC."
    )


print(
    "PASS: OSQ200 matches CDC."
)


#
# Father's age when hip fractured:
#
# 9-99 = 107
# 999  = 2
# Missing = 6109
#

s = df["OSQ210"]


if not (
    int(
        s.between(
            9,
            99
        ).sum()
    )
    == 107
    and
    int(
        s.eq(777)
        .sum()
    )
    == 0
    and
    int(
        s.eq(999)
        .sum()
    )
    == 2
    and
    int(
        s.isna()
        .sum()
    )
    == 6109
):

    raise ValueError(
        "OSQ210 differs from CDC."
    )


print(
    "PASS: OSQ210 matches CDC."
)


# ============================================================
# 23. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# IMPORTANT:
#
# OSQ_F requires NO numeric correction.
#
# Therefore all 92 numeric variables must remain
# exactly equal to the raw source XPT.
#
# Int64 conversion changes only representation:
#
#     51624.0 -> 51624
#     1.0     -> 1
#     99999.0 -> 99999
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 92 variables preserve "
    "the original XPT numeric values exactly."
)


# ============================================================
# 24. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO float_format.
# NO rounding.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 25. RE-READ EXPORTED CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 26. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


csv_failures = []


for col in source_df.columns:

    expected_values = (
        source_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 92 CSV columns match "
    "the source XPT values exactly."
)


# ============================================================
# 27. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in source_df.columns:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 28. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in csv_df.select_dtypes(
    include=[np.number]
).columns:

    tiny_csv_count += int(
        csv_df[col]
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Unexpected tiny XPORT artifacts remain "
        "in final CSV."
    )


print(
    "PASS: No tiny-value artifacts."
)


# ============================================================
# 29. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# OSQ_F contains NO genuine decimal variables.
#
# Therefore ".0" should not occur anywhere in this
# particular final CSV.
#
# IMPORTANT:
#
# Do NOT apply this global rule to Dietary,
# Lab, Examination, or other mixed-decimal datasets.
#

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count != 0:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 30. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 31. FINAL CSV SEQN CHECK
# ============================================================

print("\n--- FINAL CSV SEQN CHECK ---")


if (
    csv_df["SEQN"].min()
    != 51624
    or
    csv_df["SEQN"].max()
    != 62160
    or
    int(
        csv_df["SEQN"]
        .isna()
        .sum()
    )
    != 0
    or
    int(
        csv_df["SEQN"]
        .duplicated()
        .sum()
    )
    != 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 32. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "OSQ_F CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Tiny-value corrections: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "XPT -> CSV unexpected differences: 0 expected"
)


print(
    "Missing-value differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0: 0 expected"
)


print(
    "Scientific notation: 0 expected"
)

NHANES 2009-2010 OSQ_F XPT -> CSV
Rows: 6,218
Columns: 92
PASS: Structure = 6,218 rows x 92 columns.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny artifact occurrences: 0
PASS: No tiny XPORT zero-decoding artifacts.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in OSQ_F.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 51624
Maximum: 62160
PASS: SEQN validated.

PASS: All 92 OSQ_F variables formatted without unnecessary .0.

--- CDC FRACTURE HISTORY CHECKS ---
PASS: OSQ010A matches CDC.
PASS: OSQ010B matches CDC.
PASS: OSQ010C matches CDC.

--- CDC FRACTURE COUNT CHECKS ---
PASS: OSQ020A matches CDC.
PASS: OSQ020B matches CDC.
PASS: OSQ020C matches CDC.

--- OSD030AA ---
PASS: OSD030AA matches CDC.
PASS: OSQ040AA matches CDC.
PASS: OSD050AA matches CDC.
PASS: OSQ080 matches CDC.
PASS: OSQ060 matches CDC.
PASS: OSQ070 matches CDC.
PASS: OSQ130 matches CDC.
PASS: OSQ140Q matches CDC.
PASS: OSQ140U matches CDC.
PASS: OSQ150 matches CDC.
PASS: OSQ160